In [1]:
import mne
import numpy as np
import pandas as pd
from pathlib import Path
from mne_bids import (BIDSPath, write_raw_bids, read_raw_bids, print_dir_tree, get_entity_vals, make_report)

sample_dir = mne.datasets.sample.data_path() / "MEG" / "sample"
raw = mne.io.read_raw_fif(sample_dir / "sample_audvis_raw.fif")
raw.pick(["eeg", "eog", "stim"])      # EEG study only, like your lab's data
raw.info["line_freq"] = 60            # BIDS requires the mains frequency (US = 60 Hz)

events = mne.find_events(raw, stim_channel="STI 014")
event_id = {"auditory/left": 1, "auditory/right": 2,
            "visual/left": 3, "visual/right": 4,
            "smiley": 5, "button": 32}
print("Bad channels:", raw.info["bads"])

Opening raw data file /Users/marcocurran/Downloads/VS Code Profiles/thesis/Masters-Thesis-5th-Year/mne-learning/data/MNE-sample-data/MEG/sample/sample_audvis_raw.fif...
    Read a total of 3 projection items:
        PCA-v1 (1 x 102)  idle
        PCA-v2 (1 x 102)  idle
        PCA-v3 (1 x 102)  idle
    Range : 25800 ... 192599 =     42.956 ...   320.670 secs
Ready.
Finding events on: STI 014


/Users/marcocurran/Downloads/VS Code Profiles/thesis/Masters-Thesis-5th-Year/mne-learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


320 events found on stim channel STI 014
Event IDs: [ 1  2  3  4  5 32]
Bad channels: ['EEG 053']


In [2]:
bids_root = Path("../data/sample-BIDS")

bids_path = BIDSPath(subject="01", session="01", task="audiovisual", run="1", root=bids_root)
bids_path = write_raw_bids(raw, bids_path, events=events, event_id=event_id, overwrite=True)

print(bids_path)
print_dir_tree(bids_root, max_depth=4)

Opening raw data file /Users/marcocurran/Downloads/VS Code Profiles/thesis/Masters-Thesis-5th-Year/mne-learning/data/MNE-sample-data/MEG/sample/sample_audvis_raw.fif...
    Read a total of 3 projection items:
        PCA-v1 (1 x 102)  idle
        PCA-v2 (1 x 102)  idle
        PCA-v3 (1 x 102)  idle
    Range : 25800 ... 192599 =     42.956 ...   320.670 secs
Ready.
Writing '../data/sample-BIDS/README'...
Writing '../data/sample-BIDS/participants.tsv'...
Writing '../data/sample-BIDS/participants.json'...
Writing '../data/sample-BIDS/sub-01/ses-01/eeg/sub-01_ses-01_space-CapTrak_electrodes.tsv'...
Writing '../data/sample-BIDS/sub-01/ses-01/eeg/sub-01_ses-01_space-CapTrak_coordsystem.json'...
Writing '../data/sample-BIDS/sub-01/ses-01/eeg/sub-01_ses-01_space-CapTrak_electrodes.json'...
Used Annotations descriptions: ['auditory/left', 'auditory/right', 'button', 'smiley', 'visual/left', 'visual/right']
Writing '../data/sample-BIDS/sub-01/ses-01/eeg/sub-01_ses-01_task-audiovisual_run-1_ev

In [3]:
eeg_dir = bids_path.directory

events_tsv = pd.read_csv(bids_path.copy().update(suffix="events", extension=".tsv").fpath, sep="\t")
print(events_tsv.head())

channels_tsv = pd.read_csv(bids_path.copy().update(suffix="channels", extension=".tsv").fpath, sep="\t")
print(channels_tsv[["name", "type", "units", "status"]].head(10))
print(channels_tsv[channels_tsv["status"] == "bad"])

print((eeg_dir / bids_path.copy().update(suffix="eeg", extension=".json").basename).read_text())

      onset  duration      trial_type  value  sample
0  3.624618       0.0  auditory/right      2    2177
1  4.237323       0.0     visual/left      3    2545
2  4.946596       0.0   auditory/left      1    2971
3  5.692499       0.0    visual/right      4    3419
4  6.413426       0.0  auditory/right      2    3852
      name  type units status
0  STI 001  TRIG    µV   good
1  STI 002  TRIG    µV   good
2  STI 003  TRIG    µV   good
3  STI 004  TRIG    µV   good
4  STI 005  TRIG    µV   good
5  STI 006  TRIG    µV   good
6  STI 014  TRIG    µV   good
7  STI 015  TRIG    µV   good
8  STI 016  TRIG    µV   good
9  EEG 001   EEG    µV   good
       name type units  low_cutoff  high_cutoff           description  \
61  EEG 053  EEG    µV         0.1     172.1763  ElectroEncephaloGram   

    sampling_frequency status  status_description  
61           600.61499    bad                 NaN  
{
    "TaskName": "audiovisual",
    "Manufacturer": "Elekta",
    "PowerLineFrequency": 60.0,
    "S

In [4]:
raw_back = read_raw_bids(bids_path)
print(raw_back.annotations[:5])
print("first_samp before / after:", raw.first_samp, "/", raw_back.first_samp)

events_back, _ = mne.events_from_annotations(raw_back, event_id=event_id)
same_events = np.array_equal(events[:, 0] - raw.first_samp, events_back[:, 0] - raw_back.first_samp)
same_data = np.allclose(raw.get_data(picks="eeg"), raw_back.get_data(picks="eeg"), atol=1e-9)
print("Events identical:", same_events, "| EEG identical:", same_data)

Extracting parameters from ../data/sample-BIDS/sub-01/ses-01/eeg/sub-01_ses-01_task-audiovisual_run-1_eeg.vhdr...
Setting channel info structure...
Reading channel info from ../data/sample-BIDS/sub-01/ses-01/eeg/sub-01_ses-01_task-audiovisual_run-1_channels.tsv.
Reading electrode coords from ../data/sample-BIDS/sub-01/ses-01/eeg/sub-01_ses-01_space-CapTrak_electrodes.tsv.
<Annotations | 5 segments: auditory/left (1), auditory/right (2), ...>
first_samp before / after: 25800 / 0
Used Annotations descriptions: ['auditory/left', 'auditory/right', 'button', 'smiley', 'visual/left', 'visual/right']


/var/folders/df/s9vtwz_n2b98hrzqtdsw39qw0000gn/T/ipykernel_26993/281285104.py:1: RuntimeWarning: There are channels without locations (n/a) that are not marked as bad: ['EOG 061']
  raw_back = read_raw_bids(bids_path)
/var/folders/df/s9vtwz_n2b98hrzqtdsw39qw0000gn/T/ipykernel_26993/281285104.py:1: RuntimeWarning: Not setting position of 1 eog channel found in montage:
['EOG 061']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw_back = read_raw_bids(bids_path)


Events identical: True | EEG identical: True


In [5]:
print("Subjects:", get_entity_vals(bids_root, "subject"))
print("Tasks:", get_entity_vals(bids_root, "task"))

all_eeg = BIDSPath(root=bids_root, datatype="eeg", suffix="eeg", extension=".vhdr").match()
for p in all_eeg:
    print(p)

print(make_report(bids_root))

Subjects: ['01']
Tasks: ['audiovisual']
../data/sample-BIDS/sub-01/ses-01/eeg/sub-01_ses-01_task-audiovisual_run-1_eeg.vhdr
Summarizing participants.tsv ../data/sample-BIDS/participants.tsv...
Summarizing scans.tsv files [PosixPath('../data/sample-BIDS/sub-01/ses-01/sub-01_ses-01_scans.tsv')]...
The participant template found: sex were all unknown;
handedness were all unknown;
ages all unknown
 The [Unspecified] dataset was created by [Unspecified1], and [Unspecified2] and
conforms to BIDS version 1.9.0. This report was generated with MNE-BIDS
(https://doi.org/10.21105/joss.01896). The dataset consists of 1 participants
(sex were all unknown; handedness were all unknown; ages all unknown) and 1
recording sessions: 01. Data was recorded using an EEG system (Elekta) sampled
at 600.61 Hz with line noise at 60.0 Hz. There was 1 scan in total. Recording
durations ranged from 277.71 to 277.71 seconds (mean = 277.71, std = 0.0), for a
total of 277.71 seconds of data recorded over all scans. F